# Phase 8: BACKTRACK + DAgger Training

Phase 7 showed 98% teacher-forced accuracy on PAUSE/RESUME traces but
autoregressive inference fails (66% on 4-bit, crashes on 8-bit).

**Two fixes:**
1. **[BACKTRACK] token** — model can undo a mistake and retry
2. **DAgger training** — gradually train on model's own rollouts

**Runtime:** A100 GPU.

In [ ]:
# Cell 1: Setup
import os
if not os.path.exists('/content/recursive-transformers/src'):
    !git clone https://github.com/dhruvsyam123/recursive-transformers.git /content/recursive-transformers
else:
    !cd /content/recursive-transformers && git pull
%cd /content/recursive-transformers
!pip install -q jax[cuda12] equinox optax jaxtyping numpy pyyaml matplotlib einops

import jax
print(f'JAX devices: {jax.devices()}')
print(f'JAX backend: {jax.default_backend()}')

In [ ]:
# Cell 2: Model — PAUSE/RESUME + BACKTRACK
import jax
import jax.numpy as jnp
import equinox as eqx
import optax
import math
import numpy as np

from src.model.looped_transformer import (
    MultiHeadSelfAttention, FeedForward, RMSNorm, count_parameters,
)
from src.data.karatsuba_trace import int_to_bits, bits_to_int

# --- Token IDs ---
PAD_ID = 0; INPUT_ID = 2; SPLIT_ID = 3
SUB_MUL_0_ID = 4; SUB_MUL_1_ID = 5; SUB_MUL_2_ID = 6
ADD_ID = 7; SUB_ID = 8; COMBINE_ID = 9
OUTPUT_ID = 10; BASE_MUL_ID = 11
BIT_0_ID = 12; BIT_1_ID = 13
PAUSE_ID = 143; RESUME_ID = 144
BACKTRACK_ID = 145  # NEW
VOCAB_SIZE = 146

# Step types
STEP_INPUT=0; STEP_SPLIT=1; STEP_SUB_MUL_0=2; STEP_SUB_MUL_1=3
STEP_SUB_MUL_2=4; STEP_ADD=5; STEP_SUB=6; STEP_COMBINE=7
STEP_OUTPUT=8; STEP_BASE_MUL=9; STEP_PAUSE=10; STEP_RESUME=11
STEP_BACKTRACK=12  # NEW

TAG_TO_STEP = {
    INPUT_ID:STEP_INPUT, SPLIT_ID:STEP_SPLIT,
    SUB_MUL_0_ID:STEP_SUB_MUL_0, SUB_MUL_1_ID:STEP_SUB_MUL_1,
    SUB_MUL_2_ID:STEP_SUB_MUL_2, ADD_ID:STEP_ADD,
    SUB_ID:STEP_SUB, COMBINE_ID:STEP_COMBINE,
    OUTPUT_ID:STEP_OUTPUT, BASE_MUL_ID:STEP_BASE_MUL,
    PAUSE_ID:STEP_PAUSE, RESUME_ID:STEP_RESUME,
    BACKTRACK_ID:STEP_BACKTRACK,
}
ALL_TAG_IDS = set(TAG_TO_STEP.keys())

def bit_to_token(b): return BIT_0_ID if b == 0 else BIT_1_ID
def token_to_bit(t): return 0 if t == BIT_0_ID else 1

# --- Model (same as Phase 5/7 but VOCAB=146) ---
def sinusoidal_encoding(positions, d_model):
    pos = positions.astype(jnp.float32)
    if pos.ndim == 0: pos = pos[None]; squeeze = True
    else: squeeze = False
    dim_indices = jnp.arange(0, d_model, 2, dtype=jnp.float32)
    freqs = jnp.exp(-dim_indices * (math.log(10000.0) / d_model))
    angles = pos[:, None] * freqs[None, :]
    enc = jnp.zeros((pos.shape[0], d_model))
    enc = enc.at[:, 0::2].set(jnp.sin(angles))
    enc = enc.at[:, 1::2].set(jnp.cos(angles))
    return enc[0] if squeeze else enc

class AllSinusoidalHierarchicalPos(eqx.Module):
    d_model: int = eqx.field(static=True)
    comp_dim: int = eqx.field(static=True)
    def __init__(self, d_model):
        assert d_model % 4 == 0
        self.d_model = d_model; self.comp_dim = d_model // 4
    def __call__(self, bit_sig, depth, sub_id, step_type):
        return jnp.concatenate([
            sinusoidal_encoding(bit_sig, self.comp_dim),
            sinusoidal_encoding(depth, self.comp_dim),
            sinusoidal_encoding(sub_id, self.comp_dim),
            sinusoidal_encoding(step_type, self.comp_dim),
        ], axis=-1)

class LoopBlock(eqx.Module):
    attention: MultiHeadSelfAttention; ffn: FeedForward
    norm1: RMSNorm; norm2: RMSNorm; d_model: int = eqx.field(static=True)
    def __init__(self, d_model, n_heads, d_ff, *, key):
        k1, k2 = jax.random.split(key)
        self.attention = MultiHeadSelfAttention(d_model, n_heads, key=k1)
        self.ffn = FeedForward(d_model, d_ff, key=k2)
        self.norm1 = RMSNorm(d_model); self.norm2 = RMSNorm(d_model)
        self.d_model = d_model
    def __call__(self, x, timestep, mask=None):
        t_emb = sinusoidal_encoding(timestep, self.d_model)
        x_cond = x + t_emb[None, :]
        normed = jax.vmap(self.norm1)(x_cond)
        x = x + self.attention(normed, mask=mask)
        normed = jax.vmap(self.norm2)(x)
        x = x + jax.vmap(self.ffn)(normed)
        return x

class KaratsubaModel(eqx.Module):
    token_embed: eqx.nn.Embedding; hier_pos: AllSinusoidalHierarchicalPos
    block: LoopBlock; final_norm: RMSNorm; output_head: eqx.nn.Linear
    inject_scale: jnp.ndarray
    def __call__(self, tokens, n_loops, bit_sig, depth, sub_id, step_type):
        x0 = jax.vmap(self.token_embed)(tokens)
        pos = self.hier_pos(bit_sig, depth, sub_id, step_type)
        x0 = x0 + pos; x = x0
        seq_len = tokens.shape[0]
        mask = jnp.tril(jnp.ones((seq_len, seq_len), dtype=jnp.bool_))
        def scan_fn(hidden, timestep):
            hidden = hidden + self.inject_scale * x0
            hidden = self.block(hidden, timestep, mask=mask)
            return hidden, None
        x, _ = jax.lax.scan(scan_fn, x, jnp.arange(n_loops))
        x = jax.vmap(self.final_norm)(x)
        return jax.vmap(self.output_head)(x)

D_MODEL = 256
key = jax.random.PRNGKey(42)
k1, k2, k3 = jax.random.split(key, 3)
model = KaratsubaModel(
    token_embed=eqx.nn.Embedding(VOCAB_SIZE, D_MODEL, key=k1),
    hier_pos=AllSinusoidalHierarchicalPos(D_MODEL),
    block=LoopBlock(D_MODEL, n_heads=8, d_ff=512, key=k2),
    final_norm=RMSNorm(D_MODEL),
    output_head=eqx.nn.Linear(D_MODEL, VOCAB_SIZE, key=k3),
    inject_scale=jnp.array(0.1),
)
print(f'Parameters: {count_parameters(model):,}')

In [ ]:
# Cell 3: Trace generator + BACKTRACK augmentation + dataset + mask
import random as pyrandom

# --- Single-level trace generator (same as Phase 7) ---
def generate_single_level_trace(x, y, n_bits, base_case_bits=4):
    """Generate single-level PAUSE/RESUME trace. Returns (seq, product)."""
    assert x >= 0 and y >= 0 and x < (1 << n_bits) and y < (1 << n_bits)
    product = x * y
    product_bits = 2 * n_bits
    seq = []
    def add_tag(tag_id, step_type):
        seq.append((tag_id, {'bit_significance': 200 + step_type, 'step_type': step_type}))
    def add_bits(bits_list, step_type, bit_offset=0):
        for i, b in enumerate(bits_list):
            seq.append((bit_to_token(b), {'bit_significance': bit_offset + i, 'step_type': step_type}))
    
    if n_bits <= base_case_bits:
        add_tag(INPUT_ID, STEP_INPUT)
        add_bits(int_to_bits(x, n_bits), STEP_INPUT, bit_offset=0)
        add_bits(int_to_bits(y, n_bits), STEP_INPUT, bit_offset=n_bits)
        add_tag(BASE_MUL_ID, STEP_BASE_MUL)
        add_bits(int_to_bits(product, product_bits), STEP_BASE_MUL)
        add_tag(OUTPUT_ID, STEP_OUTPUT)
        add_bits(int_to_bits(product, product_bits), STEP_OUTPUT)
        return seq, product
    
    half = n_bits // 2; hi_bits = n_bits - half; mask_low = (1 << half) - 1
    x_lo, x_hi = x & mask_low, x >> half
    y_lo, y_hi = y & mask_low, y >> half
    z0 = x_lo * y_lo; z2 = x_hi * y_hi
    sum_x, sum_y = x_lo + x_hi, y_lo + y_hi
    z1_raw = sum_x * sum_y; z1 = z1_raw - z0 - z2
    result = z0 + (z1 << half) + (z2 << (2 * half))
    assert result == product
    z0_n, z2_n = half, hi_bits
    sum_actual_bits = max(half, hi_bits) + 1; z1_n = sum_actual_bits
    
    add_tag(INPUT_ID, STEP_INPUT)
    add_bits(int_to_bits(x, n_bits), STEP_INPUT, bit_offset=0)
    add_bits(int_to_bits(y, n_bits), STEP_INPUT, bit_offset=n_bits)
    add_tag(SPLIT_ID, STEP_SPLIT)
    off = 0
    add_bits(int_to_bits(x_hi, hi_bits), STEP_SPLIT, bit_offset=off); off += hi_bits
    add_bits(int_to_bits(x_lo, half), STEP_SPLIT, bit_offset=off); off += half
    add_bits(int_to_bits(y_hi, hi_bits), STEP_SPLIT, bit_offset=off); off += hi_bits
    add_bits(int_to_bits(y_lo, half), STEP_SPLIT, bit_offset=off)
    add_tag(SUB_MUL_0_ID, STEP_SUB_MUL_0)
    add_bits(int_to_bits(x_lo, z0_n), STEP_SUB_MUL_0, bit_offset=0)
    add_bits(int_to_bits(y_lo, z0_n), STEP_SUB_MUL_0, bit_offset=z0_n)
    add_tag(PAUSE_ID, STEP_PAUSE)
    add_tag(RESUME_ID, STEP_RESUME)
    add_bits(int_to_bits(z0, 2 * z0_n), STEP_RESUME)
    add_tag(SUB_MUL_2_ID, STEP_SUB_MUL_2)
    add_bits(int_to_bits(x_hi, z2_n), STEP_SUB_MUL_2, bit_offset=0)
    add_bits(int_to_bits(y_hi, z2_n), STEP_SUB_MUL_2, bit_offset=z2_n)
    add_tag(PAUSE_ID, STEP_PAUSE)
    add_tag(RESUME_ID, STEP_RESUME)
    add_bits(int_to_bits(z2, 2 * z2_n), STEP_RESUME)
    add_tag(ADD_ID, STEP_ADD)
    add_bits(int_to_bits(sum_x, sum_actual_bits), STEP_ADD, bit_offset=0)
    add_bits(int_to_bits(sum_y, sum_actual_bits), STEP_ADD, bit_offset=sum_actual_bits)
    add_tag(SUB_MUL_1_ID, STEP_SUB_MUL_1)
    add_bits(int_to_bits(sum_x, z1_n), STEP_SUB_MUL_1, bit_offset=0)
    add_bits(int_to_bits(sum_y, z1_n), STEP_SUB_MUL_1, bit_offset=z1_n)
    add_tag(PAUSE_ID, STEP_PAUSE)
    add_tag(RESUME_ID, STEP_RESUME)
    add_bits(int_to_bits(z1_raw, 2 * z1_n), STEP_RESUME)
    add_tag(SUB_ID, STEP_SUB)
    add_bits(int_to_bits(z1, product_bits), STEP_SUB)
    add_tag(COMBINE_ID, STEP_COMBINE)
    add_bits(int_to_bits(result, product_bits), STEP_COMBINE)
    add_tag(OUTPUT_ID, STEP_OUTPUT)
    add_bits(int_to_bits(result, product_bits), STEP_OUTPUT)
    return seq, product


# --- BACKTRACK augmentation ---
def augment_with_backtrack(seq, corruption_prob=0.3, max_corruptions=3, rng=None):
    """Insert random corruptions followed by [BACKTRACK] + correct token.
    Only corrupts PREDICTABLE bit tokens (not INPUT bits, not RESUME bits, not tags)."""
    if rng is None: rng = pyrandom.Random()
    
    # Find predictable bit positions (bits NOT after INPUT or RESUME)
    last_tag = None
    predictable_bit_indices = []
    for i, (tok_id, pos) in enumerate(seq):
        if tok_id in ALL_TAG_IDS:
            last_tag = tok_id
        elif tok_id in (BIT_0_ID, BIT_1_ID):
            if last_tag not in (INPUT_ID, RESUME_ID):
                predictable_bit_indices.append(i)
    
    if not predictable_bit_indices:
        return seq  # nothing to corrupt (e.g., very short trace)
    
    # Select positions to corrupt
    n_corruptions = 0
    corrupt_positions = set()
    for idx in predictable_bit_indices:
        if rng.random() < corruption_prob and n_corruptions < max_corruptions:
            corrupt_positions.add(idx)
            n_corruptions += 1
    
    if not corrupt_positions:
        return seq  # no corruptions this time
    
    # Build augmented sequence
    augmented = []
    for i, (tok_id, pos) in enumerate(seq):
        if i in corrupt_positions:
            # Insert wrong bit
            wrong_bit = BIT_1_ID if tok_id == BIT_0_ID else BIT_0_ID
            augmented.append((wrong_bit, dict(pos)))  # wrong token
            # Insert [BACKTRACK]
            augmented.append((BACKTRACK_ID, {'bit_significance': 200 + STEP_BACKTRACK, 'step_type': STEP_BACKTRACK}))
            # Insert correct token
            augmented.append((tok_id, dict(pos)))
        else:
            augmented.append((tok_id, dict(pos)))
    
    return augmented


# --- Dataset creation ---
def make_dataset(bit_widths_and_counts, base_case_bits=4, seed=42, augment=False, aug_prob=0.3):
    rng = np.random.RandomState(seed)
    aug_rng = pyrandom.Random(seed + 1)
    examples = []
    for n_bits, n_samples in bit_widths_and_counts:
        max_val = 1 << n_bits
        pairs = [(x, y) for x in range(max_val) for y in range(max_val)] if n_bits <= 4 \
            else [(rng.randint(0, max_val), rng.randint(0, max_val)) for _ in range(n_samples)]
        for x, y in pairs:
            seq, prod = generate_single_level_trace(x, y, n_bits, base_case_bits)
            # Clean version
            token_ids = np.array([t for t, _ in seq], dtype=np.int32)
            bit_sigs = np.array([p['bit_significance'] for _, p in seq], dtype=np.int32)
            step_types = np.array([p['step_type'] for _, p in seq], dtype=np.int32)
            examples.append((token_ids, bit_sigs, step_types, x, y, n_bits, prod))
            # Augmented version (with BACKTRACK)
            if augment:
                aug_seq = augment_with_backtrack(seq, corruption_prob=aug_prob, rng=aug_rng)
                if len(aug_seq) > len(seq):  # only if corruption happened
                    a_tids = np.array([t for t, _ in aug_seq], dtype=np.int32)
                    a_bsig = np.array([p['bit_significance'] for _, p in aug_seq], dtype=np.int32)
                    a_st = np.array([p['step_type'] for _, p in aug_seq], dtype=np.int32)
                    examples.append((a_tids, a_bsig, a_st, x, y, n_bits, prod))
    return examples


def get_batches(examples, batch_size, max_len, shuffle=True, rng_seed=None):
    rng = np.random.RandomState(rng_seed)
    indices = np.arange(len(examples))
    if shuffle: rng.shuffle(indices)
    for start in range(0, len(indices), batch_size):
        batch_idx = indices[start:start+batch_size]
        bs = len(batch_idx)
        tk = np.zeros((bs, max_len), dtype=np.int32)
        bs_arr = np.zeros((bs, max_len), dtype=np.int32)
        st_arr = np.zeros((bs, max_len), dtype=np.int32)
        pm = np.zeros((bs, max_len), dtype=np.float32)
        for i, idx in enumerate(batch_idx):
            token_ids, bit_sigs, step_types = examples[idx][:3]
            sl = min(len(token_ids), max_len)
            tk[i, :sl] = token_ids[:sl]
            bs_arr[i, :sl] = bit_sigs[:sl]
            st_arr[i, :sl] = step_types[:sl]
            pm[i, :sl] = 1.0
        yield {'token_ids': tk, 'bit_sig': bs_arr, 'step_type': st_arr, 'pad_mask': pm}


# --- Loss mask ---
def make_mask(token_ids_full, pad_mask_full):
    """Mask INPUT bits and RESUME bits. BACKTRACK token and post-BACKTRACK correct token are predictable."""
    is_bit = (token_ids_full == BIT_0_ID) | (token_ids_full == BIT_1_ID)
    is_any_tag = jnp.zeros_like(token_ids_full, dtype=jnp.bool_)
    for tid in ALL_TAG_IDS:
        is_any_tag = is_any_tag | (token_ids_full == tid)
    tag_or_zero = jnp.where(is_any_tag, token_ids_full, 0)
    def forward_fill_row(row):
        def scan_fill(carry, x):
            val = jnp.where(x > 0, x, carry)
            return val, val
        _, filled = jax.lax.scan(scan_fill, jnp.int32(0), row)
        return filled
    last_tag = jax.vmap(forward_fill_row)(tag_or_zero)
    unpredictable = is_bit & ((last_tag == INPUT_ID) | (last_tag == RESUME_ID))
    predictable = ~unpredictable
    full_mask = pad_mask_full * predictable.astype(jnp.float32)
    return full_mask[:, 1:]


# --- Build datasets ---
print('Building datasets...')
train_4bit = make_dataset([(4, 256)], augment=True, aug_prob=0.3, seed=42)
train_8bit = make_dataset([(8, 4000)], augment=True, aug_prob=0.3, seed=42)
# Clean-only for eval
eval_8bit = make_dataset([(8, 200)], augment=False, seed=999)

ml4 = max(len(ex[0]) for ex in train_4bit) + 2
ml8 = max(len(ex[0]) for ex in train_8bit) + 2
eml8 = max(len(ex[0]) for ex in eval_8bit) + 2

n_clean_4 = sum(1 for ex in train_4bit if BACKTRACK_ID not in ex[0])
n_aug_4 = len(train_4bit) - n_clean_4
n_clean_8 = sum(1 for ex in train_8bit if BACKTRACK_ID not in ex[0])
n_aug_8 = len(train_8bit) - n_clean_8

print(f'4-bit: {n_clean_4} clean + {n_aug_4} augmented, max_len={ml4}')
print(f'8-bit: {n_clean_8} clean + {n_aug_8} augmented, max_len={ml8}')

# Show augmented example
print('\nSample augmented 8-bit trace:')
for ex in train_8bit:
    if BACKTRACK_ID in ex[0]:
        tag_names = {INPUT_ID:'IN', SPLIT_ID:'SP', SUB_MUL_0_ID:'M0', SUB_MUL_1_ID:'M1',
                     SUB_MUL_2_ID:'M2', ADD_ID:'AD', SUB_ID:'SU', COMBINE_ID:'CO',
                     OUTPUT_ID:'OU', BASE_MUL_ID:'BM', PAUSE_ID:'PA', RESUME_ID:'RE',
                     BACKTRACK_ID:'BT', BIT_0_ID:'0', BIT_1_ID:'1'}
        readable = [tag_names.get(int(t), '?') for t in ex[0][:40]]
        print(' '.join(readable) + '...')
        bt_count = sum(1 for t in ex[0] if t == BACKTRACK_ID)
        print(f'  Length: {len(ex[0])} tokens, {bt_count} backtracks')
        break

In [ ]:
# Cell 4: Training — Stage 1 (teacher forced) + Stage 2 (DAgger)
import random

schedule = optax.warmup_cosine_decay_schedule(
    init_value=1e-5, peak_value=1e-3, warmup_steps=500,
    decay_steps=60000, end_value=1e-6
)
optimizer = optax.adamw(learning_rate=schedule, weight_decay=0.15)
opt_state = optimizer.init(eqx.filter(model, eqx.is_array))

def make_train_step(n_loops):
    @eqx.filter_jit
    def train_step(model, opt_state, token_ids, pad_mask, bit_sig, step_type):
        input_ids = token_ids[:, :-1]
        target_ids = token_ids[:, 1:]
        inp_bs = bit_sig[:, :-1]; inp_st = step_type[:, :-1]
        mask = make_mask(token_ids, pad_mask)
        zeros = jnp.zeros_like(inp_bs)
        def loss_fn(model):
            def fwd(ids, bs, st):
                return model(ids, n_loops, bs, zeros[0], zeros[0], st)
            logits = jax.vmap(fwd)(input_ids, inp_bs, inp_st)
            log_probs = jax.nn.log_softmax(logits, axis=-1)
            targets_oh = jax.nn.one_hot(target_ids, VOCAB_SIZE)
            tok_loss = -jnp.sum(targets_oh * log_probs, axis=-1)
            return jnp.sum(tok_loss * mask) / jnp.maximum(jnp.sum(mask), 1.0)
        loss, grads = eqx.filter_value_and_grad(loss_fn)(model)
        updates, new_opt = optimizer.update(grads, opt_state, eqx.filter(model, eqx.is_array))
        return eqx.apply_updates(model, updates), new_opt, loss
    return train_step

train_steps = {n: make_train_step(n) for n in [4, 6, 8]}

# Teacher-forced eval on clean traces
def tf_eval(model, examples, max_len, n_loops=8):
    correct, total = 0, 0
    for batch in get_batches(examples, batch_size=16, max_len=max_len, shuffle=False):
        tk = jnp.array(batch['token_ids']); pm = jnp.array(batch['pad_mask'])
        bs = jnp.array(batch['bit_sig']); st = jnp.array(batch['step_type'])
        inp = tk[:, :-1]; tgt = tk[:, 1:]
        inp_bs = bs[:, :-1]; inp_st = st[:, :-1]
        zeros = jnp.zeros_like(inp_bs)
        em = make_mask(tk, pm)
        def fwd(ids, b, s): return model(ids, n_loops, b, zeros[0], zeros[0], s)
        logits = jax.vmap(fwd)(inp, inp_bs, inp_st)
        preds = jnp.argmax(logits, axis=-1)
        matches = (preds == tgt) | (em == 0)
        correct += int(jnp.all(matches, axis=-1).sum())
        total += tk.shape[0]
    return correct, total

# ===== STAGE 1: Teacher forcing with BACKTRACK augmentation =====
rng = random.Random(42)
print('=== STAGE 1: Teacher forcing with BACKTRACK augmentation ===')
for epoch in range(300):
    epoch_loss, n_batches = 0.0, 0
    n_loops = rng.choice([4, 6, 8])
    step_fn = train_steps[n_loops]
    for batch in get_batches(train_4bit, batch_size=64, max_len=ml4):
        tk=jnp.array(batch['token_ids']); pm=jnp.array(batch['pad_mask'])
        bs=jnp.array(batch['bit_sig']); st=jnp.array(batch['step_type'])
        model, opt_state, loss = step_fn(model, opt_state, tk, pm, bs, st)
        epoch_loss += float(loss); n_batches += 1
    n_loops = rng.choice([4, 6, 8])
    step_fn = train_steps[n_loops]
    for batch in get_batches(train_8bit, batch_size=16, max_len=ml8):
        tk=jnp.array(batch['token_ids']); pm=jnp.array(batch['pad_mask'])
        bs=jnp.array(batch['bit_sig']); st=jnp.array(batch['step_type'])
        model, opt_state, loss = step_fn(model, opt_state, tk, pm, bs, st)
        epoch_loss += float(loss); n_batches += 1
    if epoch % 50 == 0:
        avg = epoch_loss / n_batches
        c8, t8 = tf_eval(model, eval_8bit, eml8)
        print(f'Epoch {epoch:4d} | Loss: {avg:.4f} | 8-bit TF: {c8}/{t8} = {c8/t8:.1%}')

print('\nStage 1 complete.')
c8, t8 = tf_eval(model, eval_8bit, eml8)
print(f'Final 8-bit TF: {c8}/{t8} = {c8/t8:.1%}')

In [ ]:
# Cell 5: Stage 2 — DAgger with partial rollouts
# Generate partial autoregressive rollouts, train on model's actual trajectory

def model_predict_next(model, token_ids, bit_sigs, step_types, n_loops=8):
    """Predict next token from current sequence."""
    tokens = jnp.array(token_ids, dtype=jnp.int32)
    bs = jnp.array(bit_sigs, dtype=jnp.int32)
    st = jnp.array(step_types, dtype=jnp.int32)
    zeros = jnp.zeros_like(bs)
    logits = model(tokens, n_loops, bs, zeros, zeros, st)
    return int(jnp.argmax(logits[-1]))


def dagger_generate_trajectory(model, clean_example, rollout_len, n_loops=8):
    """Teacher-force first K tokens, then model generates rollout_len tokens.
    Returns augmented trajectory where wrong model tokens are followed by [BACKTRACK] + correct."""
    token_ids_gt, bit_sigs_gt, step_types_gt = clean_example[:3]
    seq_len = len(token_ids_gt)
    
    # Choose where to start rollout (random position in the sequence)
    max_start = max(1, seq_len - rollout_len - 1)
    start = pyrandom.randint(1, max_start)
    
    # Build trajectory: teacher-forced prefix + model rollout
    traj_tokens = list(token_ids_gt[:start])
    traj_bsigs = list(bit_sigs_gt[:start])
    traj_stypes = list(step_types_gt[:start])
    
    # Target for each position (what oracle says should come next)
    traj_targets = list(token_ids_gt[1:start])  # targets for teacher-forced prefix
    
    # Model rollout
    for i in range(rollout_len):
        gt_pos = start + i
        if gt_pos >= seq_len - 1:
            break
        
        # Model predicts
        pred = model_predict_next(model, traj_tokens, traj_bsigs, traj_stypes, n_loops)
        gt_next = int(token_ids_gt[gt_pos])
        
        if pred == gt_next:
            # Correct! Add to trajectory
            traj_tokens.append(gt_next)
            traj_bsigs.append(int(bit_sigs_gt[gt_pos]))
            traj_stypes.append(int(step_types_gt[gt_pos]))
            traj_targets.append(int(token_ids_gt[gt_pos + 1]) if gt_pos + 1 < seq_len else PAD_ID)
        else:
            # Wrong! Insert: wrong_token, then target=[BACKTRACK]
            traj_tokens.append(pred)
            traj_bsigs.append(int(bit_sigs_gt[gt_pos]))
            traj_stypes.append(int(step_types_gt[gt_pos]))
            traj_targets.append(BACKTRACK_ID)  # model should learn to emit BACKTRACK here
            
            # Then [BACKTRACK] token, target=correct_token
            traj_tokens.append(BACKTRACK_ID)
            traj_bsigs.append(200 + STEP_BACKTRACK)
            traj_stypes.append(STEP_BACKTRACK)
            traj_targets.append(gt_next)  # after backtrack, emit correct token
            
            # Then correct token (continue from here)
            traj_tokens.append(gt_next)
            traj_bsigs.append(int(bit_sigs_gt[gt_pos]))
            traj_stypes.append(int(step_types_gt[gt_pos]))
            if gt_pos + 1 < seq_len:
                traj_targets.append(int(token_ids_gt[gt_pos + 1]))
            else:
                traj_targets.append(PAD_ID)
    
    return (np.array(traj_tokens, dtype=np.int32),
            np.array(traj_bsigs, dtype=np.int32),
            np.array(traj_stypes, dtype=np.int32),
            np.array(traj_targets, dtype=np.int32))


def dagger_train_on_trajectories(model, opt_state, trajectories, n_loops=8):
    """Train on collected DAgger trajectories (standard supervised on model's actual prefixes)."""
    if not trajectories:
        return model, opt_state, 0.0
    
    max_len = max(len(t[0]) for t in trajectories) + 1
    bs = len(trajectories)
    
    tk = np.zeros((bs, max_len), dtype=np.int32)
    bsig = np.zeros((bs, max_len), dtype=np.int32)
    st = np.zeros((bs, max_len), dtype=np.int32)
    tgt = np.zeros((bs, max_len - 1), dtype=np.int32)
    pm = np.zeros((bs, max_len), dtype=np.float32)
    tgt_mask = np.zeros((bs, max_len - 1), dtype=np.float32)
    
    for i, (t_tok, t_bs, t_st, t_tgt) in enumerate(trajectories):
        sl = len(t_tok)
        tk[i, :sl] = t_tok; bsig[i, :sl] = t_bs; st[i, :sl] = t_st
        pm[i, :sl] = 1.0
        tl = len(t_tgt)
        tgt[i, :tl] = t_tgt; tgt_mask[i, :tl] = 1.0
    
    step_fn = train_steps[n_loops]
    # Use the mask from the token sequence for standard training
    jt = jnp.array(tk); jp = jnp.array(pm)
    jb = jnp.array(bsig); js = jnp.array(st)
    model, opt_state, loss = step_fn(model, opt_state, jt, jp, jb, js)
    return model, opt_state, float(loss)


# DAgger training loop
print('=== STAGE 2: DAgger with partial rollouts ===')
rollout_schedule = [(200, 1), (225, 2), (250, 4), (275, 8), (300, 16)]  # (epoch, rollout_len)
current_rollout = 1

# Get clean 8-bit examples for DAgger
clean_8bit = [ex for ex in train_8bit if BACKTRACK_ID not in ex[0]]

for epoch in range(300, 500):
    # Update rollout length based on schedule
    for sched_epoch, sched_len in rollout_schedule:
        if epoch >= sched_epoch:
            current_rollout = sched_len
    
    epoch_loss, n_batches = 0.0, 0
    
    # Standard teacher-forced training on augmented data (50%)
    n_loops = rng.choice([4, 6, 8])
    step_fn = train_steps[n_loops]
    for batch in get_batches(train_4bit, batch_size=64, max_len=ml4):
        tk=jnp.array(batch['token_ids']); pm=jnp.array(batch['pad_mask'])
        bs=jnp.array(batch['bit_sig']); st=jnp.array(batch['step_type'])
        model, opt_state, loss = step_fn(model, opt_state, tk, pm, bs, st)
        epoch_loss += float(loss); n_batches += 1
    
    # DAgger: generate trajectories on 8-bit examples, train on them
    sample_size = min(64, len(clean_8bit))
    sampled = pyrandom.sample(clean_8bit, sample_size)
    trajectories = []
    for ex in sampled:
        traj = dagger_generate_trajectory(model, ex, current_rollout, n_loops=8)
        trajectories.append(traj)
    
    # Train on collected trajectories in mini-batches
    for i in range(0, len(trajectories), 16):
        batch_trajs = trajectories[i:i+16]
        model, opt_state, loss = dagger_train_on_trajectories(model, opt_state, batch_trajs)
        epoch_loss += float(loss); n_batches += 1
    
    if epoch % 50 == 0:
        avg = epoch_loss / max(n_batches, 1)
        c8, t8 = tf_eval(model, eval_8bit, eml8)
        print(f'Epoch {epoch:4d} | Loss: {avg:.4f} | 8-bit TF: {c8}/{t8} = {c8/t8:.1%} | rollout={current_rollout}')

print('\nStage 2 complete.')

In [ ]:
# Cell 6: Autoregressive inference with BACKTRACK + PAUSE/RESUME recursion
import time

MAX_BACKTRACKS = 3

def extract_sub_problem(token_ids):
    sub_mul_ids = {SUB_MUL_0_ID, SUB_MUL_1_ID, SUB_MUL_2_ID}
    last_sub_pos = -1
    for i in range(len(token_ids) - 1, -1, -1):
        if token_ids[i] in sub_mul_ids:
            last_sub_pos = i; break
    if last_sub_pos == -1: return None
    bits = []
    for i in range(last_sub_pos + 1, len(token_ids)):
        t = token_ids[i]
        if t == BIT_0_ID: bits.append(0)
        elif t == BIT_1_ID: bits.append(1)
        elif t in ALL_TAG_IDS: break
    if len(bits) == 0 or len(bits) % 2 != 0: return None
    half = len(bits) // 2
    return bits_to_int(bits[:half]), bits_to_int(bits[half:]), half


def recursive_inference(model, x, y, n_bits, base_case_bits=4, n_loops=8, depth=0):
    x_bits = int_to_bits(x, n_bits); y_bits = int_to_bits(y, n_bits)
    token_ids = [INPUT_ID]
    bit_sigs = [200 + STEP_INPUT]
    step_types = [STEP_INPUT]
    for i, b in enumerate(x_bits):
        token_ids.append(bit_to_token(b)); bit_sigs.append(i); step_types.append(STEP_INPUT)
    for i, b in enumerate(y_bits):
        token_ids.append(bit_to_token(b)); bit_sigs.append(i); step_types.append(STEP_INPUT)
    
    total_tokens = len(token_ids)
    max_gen = 400; current_step_type = STEP_INPUT; current_bit_idx = 0
    backtrack_count = 0; total_backtracks = 0
    
    for _ in range(max_gen):
        next_tok = model_predict_next(model, token_ids, bit_sigs, step_types, n_loops)
        
        if next_tok == BACKTRACK_ID:
            if backtrack_count < MAX_BACKTRACKS and len(token_ids) > 1:
                token_ids.pop(); bit_sigs.pop(); step_types.pop()
                backtrack_count += 1; total_backtracks += 1
                continue
            else:
                continue  # ignore excessive backtracks
        
        backtrack_count = 0  # reset on non-backtrack token
        
        if next_tok == PAUSE_ID:
            sub = extract_sub_problem(token_ids)
            if sub is None: return None, total_tokens, total_backtracks
            a, b, sub_n_bits = sub
            padded = 1
            while padded < sub_n_bits: padded <<= 1
            padded = max(padded, 4)
            sub_result, sub_tok, sub_bt = recursive_inference(model, a, b, padded, base_case_bits, n_loops, depth+1)
            total_tokens += sub_tok; total_backtracks += sub_bt
            if sub_result is None: return None, total_tokens, total_backtracks
            token_ids.append(PAUSE_ID); bit_sigs.append(200+STEP_PAUSE); step_types.append(STEP_PAUSE)
            result_n_bits = 2 * sub_n_bits
            result_bits = int_to_bits(sub_result, result_n_bits)
            token_ids.append(RESUME_ID); bit_sigs.append(200+STEP_RESUME); step_types.append(STEP_RESUME)
            for i, b in enumerate(result_bits):
                token_ids.append(bit_to_token(b)); bit_sigs.append(i); step_types.append(STEP_RESUME)
            current_step_type = STEP_RESUME; current_bit_idx = 0
            continue
        
        elif next_tok == OUTPUT_ID:
            token_ids.append(OUTPUT_ID); bit_sigs.append(200+STEP_OUTPUT); step_types.append(STEP_OUTPUT)
            product_n_bits = 2 * n_bits
            output_bits = []
            for i in range(product_n_bits):
                nb = model_predict_next(model, token_ids, bit_sigs, step_types, n_loops)
                if nb == BACKTRACK_ID and len(output_bits) > 0:
                    token_ids.pop(); bit_sigs.pop(); step_types.pop()
                    output_bits.pop(); total_backtracks += 1; continue
                token_ids.append(nb); bit_sigs.append(i); step_types.append(STEP_OUTPUT)
                output_bits.append(token_to_bit(nb))
            total_tokens += len(token_ids)
            return bits_to_int(output_bits), total_tokens, total_backtracks
        
        else:
            if next_tok in ALL_TAG_IDS:
                current_step_type = TAG_TO_STEP.get(next_tok, current_step_type)
                current_bit_idx = 0; bit_sigs.append(200 + current_step_type)
            else:
                bit_sigs.append(current_bit_idx); current_bit_idx += 1
            token_ids.append(next_tok); step_types.append(current_step_type)
    
    return None, total_tokens, total_backtracks


# Test on 4-bit and 8-bit
print('=== Autoregressive Test: 4-bit ===')
rng_test = pyrandom.Random(123)
correct, total, tot_bt = 0, 0, 0
for _ in range(50):
    a = rng_test.randint(0, 15); b = rng_test.randint(0, 15)
    result, _, bt = recursive_inference(model, a, b, 4)
    if result == a * b: correct += 1
    tot_bt += bt; total += 1
print(f'4-bit: {correct}/{total} = {correct/total:.1%}, avg backtracks: {tot_bt/total:.1f}')

print('\n=== Autoregressive Test: 8-bit ===')
correct, total, tot_bt = 0, 0, 0
for _ in range(50):
    a = rng_test.randint(0, 255); b = rng_test.randint(0, 255)
    result, _, bt = recursive_inference(model, a, b, 8)
    if result == a * b: correct += 1
    elif total < 3: print(f'  Error: {a}*{b}={a*b}, got {result}')
    tot_bt += bt; total += 1
print(f'8-bit: {correct}/{total} = {correct/total:.1%}, avg backtracks: {tot_bt/total:.1f}')

In [ ]:
# Cell 7: Full evaluation on 8/16/32/64-bit
print('=' * 65)
print('PHASE 8: BACKTRACK + DAgger — AUTOREGRESSIVE RECURSIVE INFERENCE')
print('=' * 65)

results = {}
for n_bits in [8, 16, 32]:
    max_val = 1 << n_bits
    n_samples = 100 if n_bits <= 16 else 30
    rng_eval = pyrandom.Random(999)
    correct, total, tot_bt, errors = 0, 0, 0, []
    t0 = time.time()
    for _ in range(n_samples):
        a = rng_eval.randint(0, max_val - 1); b = rng_eval.randint(0, max_val - 1)
        result, _, bt = recursive_inference(model, a, b, n_bits)
        if result == a * b: correct += 1
        else: errors.append((a, b, a*b, result))
        tot_bt += bt; total += 1
    elapsed = time.time() - t0
    acc = correct / total
    results[n_bits] = acc
    print(f'\n{n_bits:3d}-bit: {correct}/{total} = {acc:.1%}  ({elapsed:.0f}s, avg BT: {tot_bt/total:.1f})')
    if errors[:2]:
        for a, b, exp, got in errors[:2]:
            print(f'  Error: {a}*{b}={exp}, got {got}')

print('\n' + '=' * 65)
print('COMPARISON')
print('=' * 65)
print(f"{'Method':<42} {'8-bit':>8} {'16-bit':>8} {'32-bit':>8}")
print('-' * 66)
print(f"{'Phase 5: Flat sequence':<42} {'99%':>8} {'0%':>8} {'N/A':>8}")
print(f"{'Phase 6: Teacher-forced recursive':<42} {'100%':>8} {'100%':>8} {'100%':>8}")
print(f"{'Phase 7: Autoregressive (no BACKTRACK)':<42} {'66%':>8} {'N/A':>8} {'N/A':>8}")
print(f"{'Phase 8: BACKTRACK + DAgger':<42}", end='')
for nb in [8, 16, 32]:
    print(f'{results.get(nb, 0):>7.1%} ', end='')
print()
print('=' * 65)

In [ ]:
# Cell 8: Save model
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os
CKPT_DIR = '/content/drive/MyDrive/karatsuba_checkpoints/'
os.makedirs(CKPT_DIR, exist_ok=True)
eqx.tree_serialise_leaves(os.path.join(CKPT_DIR, 'model_phase8_backtrack_dagger.eqx'), model)
print(f'Model saved to {CKPT_DIR}')